# RAGAS Evaluation

Evaluate the production RAG pipeline using RAGAS (Retrieval Augmented Generation Assessment) metrics:
- **Faithfulness** – Is the generated answer factually grounded in the retrieved contexts?
- **Answer Relevancy** – How relevant is the generated answer to the user's question?
- **Context Precision** – Are the retrieved documents precise and relevant?
- **Context Recall** – Do the retrieved documents cover all needed information?

In [ ]:
import sys
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
)

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation.ragas_evaluator import RagasEvaluator  # noqa: E402

## 1. Load the Evaluation Dataset

The dataset should be a JSON array with `question` and `reference` fields:

In [ ]:
DATASET_PATH = "evaluation/datasets/rag_eval.json"
OUTPUT_DIR = "evaluation/results"

samples = RagasEvaluator.load_dataset(DATASET_PATH)

print(f"Loaded {len(samples)} evaluation samples")
for i, s in enumerate(samples):
    print(f"  [{i+1}] {s['question'][:60]}...")

## 2. Create the Evaluator

The `RagasEvaluator.from_project()` factory wraps the project's `RAGPipeline` so every call runs a full RAG query.

In [ ]:
evaluator = RagasEvaluator.from_project(
    model_name="gemini-2.0-flash",
    retrieval_top_k=20,
    rerank_top_k=5,
)

print(f"Scorers configured: {list(evaluator.scorers.keys())}")

## 3. Run Evaluation

This runs each sample through the full RAG pipeline and scores every metric.

In [ ]:
report = evaluator.evaluate(samples)

print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"Samples evaluated: {report['sample_count']}")
for metric, score in report['summary'].items():
    print(f"  {metric:20}: {score:.4f}")
print("=" * 50)

## 4. Save Reports

Results are saved as both JSON (full detail) and CSV (tabular, for Excel/analysis).

In [ ]:
json_path, csv_path = RagasEvaluator.save(report, OUTPUT_DIR)

print(f"JSON report: {json_path}")
print(f"CSV report:  {csv_path}")

# Preview first sample
print("\n--- First Sample Scores ---")
first = report['samples'][0]
for k, v in first.items():
    print(f"  {k:20}: {v}")